# Lab 10 — Explainable Cyber ML: Global and Local Explanations

*AI Cybersecurity Masterclass · Lab 10 · AI60secLab*

**Research question:** Which telemetry fields does the detector rely on overall, and why did it flag *this specific* event?

Two views, two jobs:
- **Global** (permutation importance): what the model relies on across the whole test set.
- **Local** (per-alert occlusion): what drove the score for one alert, turned into one sentence an analyst can act on.

Both measure model reliance under the tested data, not the cause of an attack.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import matplotlib.pyplot as plt

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import random_forest_pipeline
from src.explainability import permutation_importance_frame

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

model = random_forest_pipeline(X_train)
model.fit(X_train, y_train)

In [ ]:
f1_importance = permutation_importance_frame(
    model,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    max_samples=0.25,
)

display(f1_importance.head(20).round(5))

In [ ]:
top = f1_importance.head(15).sort_values("importance_mean")

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    top["feature"],
    top["importance_mean"],
    xerr=top["importance_std"],
)
ax.set_xlabel("F1 decrease after shuffling")
ax.set_title("Held-out permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
recall_importance = permutation_importance_frame(
    model,
    X_test,
    y_test,
    scoring="recall",
    n_repeats=5,
    random_state=42,
    max_samples=0.25,
)

comparison = (
    f1_importance[["feature", "importance_mean"]]
    .rename(columns={"importance_mean": "f1_importance"})
    .merge(
        recall_importance[["feature", "importance_mean"]]
        .rename(columns={"importance_mean": "recall_importance"}),
        on="feature",
        how="outer",
    )
    .fillna(0)
)

display(
    comparison
    .sort_values("recall_importance", ascending=False)
    .head(20)
    .round(5)
)

## Step 2: Local explanation for one alert

Global importance cannot tell an analyst why *this* event was flagged. A simple local method that works with any model is **occlusion**: replace one feature at a time with a neutral reference value (training median or most common category) and measure how much the attack score drops.

A large drop means the score for this event depended on that value.

In [ ]:
from src.explainability import (
    reference_row,
    local_occlusion_explanation,
    alert_explanation_sentence,
)

reference = reference_row(X_train)

test_scores = model.predict_proba(X_test)[:, 1]
attack_categories = test["attack_cat"].reset_index(drop=True)

# Pick one confidently detected Reconnaissance event to explain.
candidates = [
    i for i in range(len(X_test))
    if attack_categories[i] == "Reconnaissance" and test_scores[i] >= 0.9
]
alert_index = candidates[0]

base_score, local = local_occlusion_explanation(
    model,
    X_test.iloc[alert_index],
    reference,
)

print("Alert score:", round(base_score, 3))
display(local.head(10).round(4))

## Step 3: Turn the explanation into an analyst sentence

A useful alert says what was predicted, how strongly, which signals mattered, and what to check next. The ATT&CK-style context is a hypothesis, not a finding.

In [ ]:
sentence = alert_explanation_sentence(
    base_score,
    local,
    threshold=0.5,
    top_n=3,
    context_hint="reconnaissance-like behavior",
)
print(sentence)

## Step 4: Does local agree with global?

Compare the top local drivers for several alerts with the global ranking. Disagreement is normal: global importance averages over many events, local explanation describes one.

In [ ]:
import pandas as pd

rows = []
for category in ["Reconnaissance", "DoS", "Exploits", "Fuzzers", "Generic"]:
    idx = [
        i for i in range(len(X_test))
        if attack_categories[i] == category and test_scores[i] >= 0.9
    ]
    if not idx:
        continue
    score, frame = local_occlusion_explanation(model, X_test.iloc[idx[0]], reference)
    rows.append({
        "attack_cat": category,
        "alert_score": round(score, 3),
        "top_local_drivers": ", ".join(frame["feature"].head(3)),
    })

local_summary = pd.DataFrame(rows)
display(local_summary)
print("Top global features:", ", ".join(f1_importance["feature"].head(5)))

## Security interpretation

For the top global features, ask:
- what telemetry does the feature represent?
- could an attacker influence it?
- would it remain available in production?
- could correlated features share the same signal?
- does importance change when scoring recall instead of F1?

For the local explanation:
- would an analyst understand the sentence in ten seconds?
- does the explanation point to telemetry they can actually check?
- why is "consistent with reconnaissance" weaker than "is reconnaissance", and why is that the honest wording?

**Takeaway:** explainability shows what the model relied on. It never replaces evaluation, and it never proves the cause of an attack.